# Module 4 Solutions: Calculus — Multivariable

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
x, y, z = sp.symbols('x y z')

In [ ]:
# Ex 1
f = x**3 * y**2 + sp.exp(x*y)
print(f"∂f/∂x = {sp.diff(f, x)}")
print(f"∂f/∂y = {sp.diff(f, y)}")

In [ ]:
# Ex 2
f = x**2 + 2*y**2 + 3*z**2
grad = [sp.diff(f, v) for v in [x, y, z]]
print(f"∇f = {grad}")
print(f"At (1,1,1): {[g.subs([(x,1),(y,1),(z,1)]) for g in grad]}")

In [ ]:
# Ex 3
f = x**2 + x*y
grad_f = np.array([float(sp.diff(f,x).subs([(x,1),(y,2)])), float(sp.diff(f,y).subs([(x,1),(y,2)]))])
u = np.array([3/5, 4/5])
print(f"D_u f = {np.dot(grad_f, u)}")

In [ ]:
# Ex 4
f = sp.sin(x)*sp.cos(y) + x**2 * y**3
fxy = sp.diff(f, x, y)
fyx = sp.diff(f, y, x)
print(f"f_xy = {fxy}")
print(f"f_yx = {fyx}")
print(f"Equal? {sp.simplify(fxy - fyx) == 0}")

In [ ]:
# Ex 5
f = x**4 + y**4 - 2*x**2*y**2
H = sp.Matrix([[sp.diff(f,x,x), sp.diff(f,x,y)],[sp.diff(f,y,x), sp.diff(f,y,y)]])
H0 = H.subs([(x,0),(y,0)])
print(f"H at (0,0) = {H0}")
print(f"Eigenvalues: {np.linalg.eigvalsh(np.array(H0.tolist(), dtype=float))}")
print("Both zero → inconclusive (degenerate)")

In [ ]:
# Ex 6
f1, f2, f3 = x**2 - y**2, 2*x*y, x + y
J = sp.Matrix([[sp.diff(fi, v) for v in [x,y]] for fi in [f1,f2,f3]])
print(f"J =\n{J}")
print(f"At (1,1):\n{J.subs([(x,1),(y,1)])}")

In [ ]:
# Ex 7
def numerical_grad(f, point, h=1e-7):
    grad = np.zeros_like(point)
    for i in range(len(point)):
        p_plus = point.copy(); p_plus[i] += h
        p_minus = point.copy(); p_minus[i] -= h
        grad[i] = (f(*p_plus) - f(*p_minus)) / (2*h)
    return grad

f_np = lambda x,y,z: x**2*y + y*z**3
print(f"∇f at (1,2,3) = {numerical_grad(f_np, np.array([1.0, 2.0, 3.0]))}")

In [ ]:
# Ex 8: Lagrange: max/min xy on x²+y²=4
lam = sp.Symbol('lambda')
eqs = [y - 2*lam*x, x - 2*lam*y, x**2 + y**2 - 4]
sols = sp.solve(eqs, [x, y, lam])
for s in sols:
    print(f"({s[0]},{s[1]}): f = {s[0]*s[1]}")

In [ ]:
# Ex 9: Critical points of x³ + y³ - 3xy
f = x**3 + y**3 - 3*x*y
fx, fy = sp.diff(f,x), sp.diff(f,y)
cps = sp.solve([fx, fy], [x, y])
H = sp.Matrix([[sp.diff(f,x,x), sp.diff(f,x,y)],[sp.diff(f,y,x), sp.diff(f,y,y)]])
for cp in cps:
    H_cp = H.subs([(x,cp[0]),(y,cp[1])])
    det_H = H_cp.det()
    trace_H = H_cp.trace()
    nature = 'saddle' if det_H < 0 else ('min' if trace_H > 0 else 'max')
    print(f"({cp[0]},{cp[1]}): det(H)={det_H}, tr(H)={trace_H} → {nature}")

In [ ]:
# Ex 10: 2D GD on elongated bowl
f_np = lambda p: (p[0]-1)**2 + 10*(p[1]-2)**2
grad_np = lambda p: np.array([2*(p[0]-1), 20*(p[1]-2)])

p = np.array([0.0, 0.0])
path = [p.copy()]
for _ in range(100):
    p = p - 0.05 * grad_np(p)
    path.append(p.copy())
path = np.array(path)

X, Y = np.meshgrid(np.linspace(-1,3,100), np.linspace(-1,4,100))
Z = (X-1)**2 + 10*(Y-2)**2
plt.contour(X, Y, Z, levels=20, cmap='viridis')
plt.plot(path[:,0], path[:,1], 'ro-', ms=3, lw=1)
plt.title('GD on Elongated Bowl'); plt.grid(True); plt.show()

In [ ]:
# Ex 11: Numerical Hessian
def numerical_hessian(f, point, h=1e-5):
    n = len(point)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            pp = point.copy(); pp[i] += h; pp[j] += h
            pm = point.copy(); pm[i] += h; pm[j] -= h
            mp = point.copy(); mp[i] -= h; mp[j] += h
            mm = point.copy(); mm[i] -= h; mm[j] -= h
            H[i,j] = (f(*pp) - f(*pm) - f(*mp) + f(*mm)) / (4*h**2)
    return H

f_np = lambda x, y: np.sin(x)*np.cos(y) + x**2
print(f"H at (1,1):\n{np.round(numerical_hessian(f_np, np.array([1.0, 1.0])), 4)}")

In [ ]:
# Ex 12: Backprop from scratch
sigmoid = lambda x: 1/(1+np.exp(-x))
relu = lambda x: np.maximum(0, x)

x_val, y_val = 1.5, 1.0
w1, b1, w2, b2 = 0.5, 0.1, 0.8, -0.2

# Forward
z1 = w1*x_val + b1
a1 = relu(z1)
z2 = w2*a1 + b2
a2 = sigmoid(z2)
L = (a2 - y_val)**2

# Backward
dL_da2 = 2*(a2 - y_val)
da2_dz2 = a2*(1-a2)
dz2_dw2 = a1
dz2_db2 = 1
dz2_da1 = w2
da1_dz1 = 1.0 if z1 > 0 else 0.0
dz1_dw1 = x_val
dz1_db1 = 1

dL_dw2 = dL_da2 * da2_dz2 * dz2_dw2
dL_db2 = dL_da2 * da2_dz2 * dz2_db2
dL_dw1 = dL_da2 * da2_dz2 * dz2_da1 * da1_dz1 * dz1_dw1
dL_db1 = dL_da2 * da2_dz2 * dz2_da1 * da1_dz1 * dz1_db1
print(f"∂L/∂w1={dL_dw1:.6f}, ∂L/∂w2={dL_dw2:.6f}, ∂L/∂b1={dL_db1:.6f}, ∂L/∂b2={dL_db2:.6f}")

In [ ]:
# Ex 13: Softmax Jacobian
z_val = np.array([1.0, 2.0, 3.0])
s = np.exp(z_val) / np.exp(z_val).sum()
J = np.diag(s) - np.outer(s, s)
print(f"Jacobian:\n{np.round(J, 6)}")
print(f"Row sums: {np.round(J.sum(axis=1), 10)}")

In [ ]:
# Ex 14: GD vs Newton on Rosenbrock
def rosen(p): return 100*(p[1]-p[0]**2)**2 + (1-p[0])**2
def rosen_g(p): return np.array([-400*p[0]*(p[1]-p[0]**2)-2*(1-p[0]), 200*(p[1]-p[0]**2)])
def rosen_h(p):
    return np.array([[1200*p[0]**2-400*p[1]+2, -400*p[0]], [-400*p[0], 200]])

# GD
p = np.array([-1.0, 1.0])
gd_path = [p.copy()]
for _ in range(1000):
    p = p - 0.001 * rosen_g(p)
    gd_path.append(p.copy())
gd_path = np.array(gd_path)

# Newton
p = np.array([-1.0, 1.0])
n_path = [p.copy()]
for _ in range(50):
    H = rosen_h(p)
    try:
        p = p - np.linalg.solve(H, rosen_g(p))
    except: break
    n_path.append(p.copy())
n_path = np.array(n_path)

X, Y = np.meshgrid(np.linspace(-2,2,200), np.linspace(-1,3,200))
plt.contour(X, Y, np.log(100*(Y-X**2)**2+(1-X)**2+1), levels=30, cmap='viridis')
plt.plot(gd_path[:,0], gd_path[:,1], 'r.-', ms=1, lw=0.5, label='GD')
plt.plot(n_path[:,0], n_path[:,1], 'b.-', ms=3, lw=1, label='Newton')
plt.legend(); plt.title('GD vs Newton'); plt.grid(True); plt.show()

In [ ]:
# Ex 15: Change of variables
r = sp.Symbol('r', positive=True)
theta = sp.Symbol('theta')
for R_val in [1, 2, 5]:
    result = sp.integrate(r * sp.exp(-r**2), (r, 0, R_val), (theta, 0, 2*sp.pi))
    print(f"R={R_val}: {float(result):.8f}")
result_inf = sp.integrate(r * sp.exp(-r**2), (r, 0, sp.oo), (theta, 0, 2*sp.pi))
print(f"R=∞: {result_inf} = {float(result_inf):.8f}")